In [30]:

"""
Kaggle EV Dataset Analysis for Data Warehouse Schema Design
Performs data quality checks, explores structure, and suggests schema design.
"""

import pandas as pd
import numpy as np
from datetime import datetime
import os
from IPython.display import display


In [31]:


# Preview Kaggle EV Charging Patterns Dataset

# Load the dataset
df = pd.read_csv("data/external/ev_charging_patterns.csv")

# Display basic information
print("📈 Data Shape:", df.shape)



📈 Data Shape: (1320, 20)


In [32]:
print("\n📋 Columns and Data Types:")
print(df.dtypes.to_string())




📋 Columns and Data Types:
User ID                                      object
Vehicle Model                                object
Battery Capacity (kWh)                      float64
Charging Station ID                          object
Charging Station Location                    object
Charging Start Time                          object
Charging End Time                            object
Energy Consumed (kWh)                       float64
Charging Duration (hours)                   float64
Charging Rate (kW)                          float64
Charging Cost (USD)                         float64
Time of Day                                  object
Day of Week                                  object
State of Charge (Start %)                   float64
State of Charge (End %)                     float64
Distance Driven (since last charge) (km)    float64
Temperature (°C)                            float64
Vehicle Age (years)                         float64
Charger Type                         

In [33]:
print("\n🔎 Sample Rows:")
print(df.head(10).to_string(index=False))




🔎 Sample Rows:
User ID Vehicle Model  Battery Capacity (kWh) Charging Station ID Charging Station Location Charging Start Time   Charging End Time  Energy Consumed (kWh)  Charging Duration (hours)  Charging Rate (kW)  Charging Cost (USD) Time of Day Day of Week  State of Charge (Start %)  State of Charge (End %)  Distance Driven (since last charge) (km)  Temperature (°C)  Vehicle Age (years)    Charger Type              User Type
 User_1        BMW i3              108.463007         Station_391                   Houston 2024-01-01 00:00:00 2024-01-01 00:39:00              60.712346                   0.591363           36.389181            13.087717     Evening     Tuesday                  29.371576                86.119962                                293.602111         27.947953             2.000000 DC Fast Charger               Commuter
 User_2  Hyundai Kona              100.000000         Station_428             San Francisco 2024-01-01 01:00:00 2024-01-01 03:01:00              1

In [34]:
print("\n🔍 Missing Values Summary:")
print(df.isnull().sum().to_string())


🔍 Missing Values Summary:
User ID                                      0
Vehicle Model                                0
Battery Capacity (kWh)                       0
Charging Station ID                          0
Charging Station Location                    0
Charging Start Time                          0
Charging End Time                            0
Energy Consumed (kWh)                       66
Charging Duration (hours)                    0
Charging Rate (kW)                          66
Charging Cost (USD)                          0
Time of Day                                  0
Day of Week                                  0
State of Charge (Start %)                    0
State of Charge (End %)                      0
Distance Driven (since last charge) (km)    66
Temperature (°C)                             0
Vehicle Age (years)                          0
Charger Type                                 0
User Type                                    0


## 1. Load and Inspect Data
Load the EV charging dataset from CSV and display basic information about its structure, size, and memory usage.

In [35]:
# Load the Kaggle EV dataset
print("Loading Kaggle EV Charging Dataset")
print("=" * 60)

try:
    # Load dataset from external folder
    df = pd.read_csv("data/external/ev_charging_patterns.csv")
    
    # Print general dataset info
    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
except FileNotFoundError:
    print("File not found: data/external/ev_charging_patterns.csv")

Loading Kaggle EV Charging Dataset
Dataset Shape: 1320 rows, 20 columns
Memory Usage: 0.96 MB


## 2. Data Quality Analysis
Examine data completeness, check for missing values, analyze data types, and identify duplicate records to assess overall data quality.

In [36]:
# Check dataset quality: missing values, data types, duplicates, keys
print("\nData Quality Analysis")
print("-" * 40)

# Missing values by column
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
print("Missing Values by Column:")
for col, missing, percent in zip(df.columns, missing_data, missing_percent):
    print(f"  {col:<30}: {missing:>4} ({percent:>5.1f}%)")

# Data types of columns
print("\nData Types:")
for col, dtype in df.dtypes.items():
    print(f"  {col:<30}: {dtype}")

# Count duplicate rows
duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates}")

# Check composite key uniqueness (User ID + Charging Start Time)
composite_key_duplicates = df.duplicated(subset=['User ID', 'Charging Start Time']).sum()
print(f"Duplicate User+StartTime combinations: {composite_key_duplicates}")


Data Quality Analysis
----------------------------------------
Missing Values by Column:
  User ID                       :    0 (  0.0%)
  Vehicle Model                 :    0 (  0.0%)
  Battery Capacity (kWh)        :    0 (  0.0%)
  Charging Station ID           :    0 (  0.0%)
  Charging Station Location     :    0 (  0.0%)
  Charging Start Time           :    0 (  0.0%)
  Charging End Time             :    0 (  0.0%)
  Energy Consumed (kWh)         :   66 (  5.0%)
  Charging Duration (hours)     :    0 (  0.0%)
  Charging Rate (kW)            :   66 (  5.0%)
  Charging Cost (USD)           :    0 (  0.0%)
  Time of Day                   :    0 (  0.0%)
  Day of Week                   :    0 (  0.0%)
  State of Charge (Start %)     :    0 (  0.0%)
  State of Charge (End %)       :    0 (  0.0%)
  Distance Driven (since last charge) (km):   66 (  5.0%)
  Temperature (°C)              :    0 (  0.0%)
  Vehicle Age (years)           :    0 (  0.0%)
  Charger Type                  :   

## 3. Key Fields Analysis
Analyze dimensional attributes (User IDs, Vehicle Models, Charging Stations, User Types) to understand cardinality and distribution for star schema design.

In [37]:
# Analyze key fields that will serve as dimensions in the star schema
print("\nKey Field Analysis (Future Dimensions)")
print("-" * 40)

# User ID distribution (potential DIM_USER)
print("\nUser IDs:")
print(f"  Unique: {df['User ID'].nunique()}")
print(f"  Range: {df['User ID'].min()} to {df['User ID'].max()}")

# Vehicle Model distribution (potential DIM_VEHICLE)
print("\nVehicle Models:")
print(f"  Unique: {df['Vehicle Model'].nunique()}")
print(f"  Top 5 models:\n{df['Vehicle Model'].value_counts().head().to_string()}")

# Station ID distribution (potential DIM_STATION)
print("\nCharging Stations:")
print(f"  Unique: {df['Charging Station ID'].nunique()}")
print(f"  Top 5 stations:\n{df['Charging Station ID'].value_counts().head().to_string()}")

# User Type (potential attribute in DIM_USER)
print("\nUser Type Distribution:")
print(df['User Type'].value_counts().to_string())


Key Field Analysis (Future Dimensions)
----------------------------------------

User IDs:
  Unique: 1320
  Range: User_1 to User_999

Vehicle Models:
  Unique: 5
  Top 5 models:
Vehicle Model
Tesla Model 3    280
Hyundai Kona     266
Nissan Leaf      260
BMW i3           258
Chevy Bolt       256

Charging Stations:
  Unique: 462
  Top 5 stations:
Charging Station ID
Station_108    9
Station_17     7
Station_97     7
Station_74     7
Station_10     7

User Type Distribution:
User Type
Commuter                  476
Long-Distance Traveler    437
Casual Driver             407


## 4. Numeric Fields Analysis
Calculate descriptive statistics (mean, min, max, std dev) for all numeric measures that will become fact table metrics.

In [38]:
# Descriptive statistics for numeric measures (future FACT metrics)
print("\nNumeric Field Statistics")
print("-" * 40)

numeric_cols = [
    'Energy Consumed (kWh)',
    'Charging Duration (hours)',
    'Charging Rate (kW)',
    'Charging Cost (USD)',
    'State of Charge (Start %)',
    'State of Charge (End %)',
    'Distance Driven (since last charge) (km)',
    'Temperature (°C)',
    'Battery Capacity (kWh)',
    'Vehicle Age (years)'
]

for col in numeric_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"  Mean: {df[col].mean():.2f}")
        print(f"  Min: {df[col].min():.2f}")
        print(f"  Max: {df[col].max():.2f}")
        print(f"  Std Dev: {df[col].std():.2f}")

# Overall summary
print("\n\nOverall Numeric Summary:")
display(df[numeric_cols].describe())


Numeric Field Statistics
----------------------------------------

Energy Consumed (kWh):
  Mean: 42.64
  Min: 0.05
  Max: 152.24
  Std Dev: 22.41

Charging Duration (hours):
  Mean: 2.27
  Min: 0.10
  Max: 7.64
  Std Dev: 1.06

Charging Rate (kW):
  Mean: 25.96
  Min: 1.47
  Max: 97.34
  Std Dev: 14.01

Charging Cost (USD):
  Mean: 22.55
  Min: 0.23
  Max: 69.41
  Std Dev: 10.75

State of Charge (Start %):
  Mean: 49.13
  Min: 2.33
  Max: 152.49
  Std Dev: 24.07

State of Charge (End %):
  Mean: 75.14
  Min: 7.60
  Max: 177.71
  Std Dev: 17.08

Distance Driven (since last charge) (km):
  Mean: 153.60
  Min: 0.86
  Max: 398.36
  Std Dev: 86.00

Temperature (°C):
  Mean: 15.26
  Min: -10.72
  Max: 73.17
  Std Dev: 14.83

Battery Capacity (kWh):
  Mean: 74.53
  Min: 1.53
  Max: 193.00
  Std Dev: 20.63

Vehicle Age (years):
  Mean: 3.61
  Min: 0.00
  Max: 11.69
  Std Dev: 2.31


Overall Numeric Summary:


,Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Battery Capacity (kWh),Vehicle Age (years)
count,1254.000000,1320.000000,1254.000000,1320.000000,1320.000000,1320.000000,1254.000000,1320.000000,1320.000000,1320.000000
mean,42.642894,2.269377,25.963003,22.551352,49.130012,75.141590,153.596788,15.263591,74.534692,3.612843
std,22.411705,1.061037,14.011326,10.751494,24.074134,17.080580,86.004987,14.831216,20.626914,2.309824
min,0.045772,0.095314,1.472549,0.234317,2.325959,7.604224,0.862361,-10.724770,1.532807,0.000000
25%,23.881193,1.397623,13.856583,13.368141,27.786903,62.053266,79.445335,2.800664,62.000000,2.000000
50%,42.691405,2.258136,25.603799,22.076360,48.241771,75.682496,152.259867,14.630846,75.000000,4.000000
75%,61.206218,3.112806,37.502998,31.646044,69.277921,88.201370,226.073284,27.981810,85.000000,6.000000
max,152.238758,7.635145,97.342255,69.407743,152.489761,177.708666,398.364775,73.169588,193.003074,11.688592


## 5. DateTime Fields Analysis
Parse and analyze temporal data to understand date ranges, time-of-day patterns, and day-of-week distributions for time dimension design.

In [39]:
# Examine date and time fields (for DIM_TIME and FACT table)
print("\nDateTime Field Analysis")
print("-" * 40)

# Parse timestamps
df['start_dt'] = pd.to_datetime(df['Charging Start Time'], errors='coerce')
df['end_dt'] = pd.to_datetime(df['Charging End Time'], errors='coerce')

print("\nCharging Start Time:")
print(f"  Earliest: {df['start_dt'].min()}")
print(f"  Latest: {df['start_dt'].max()}")
print(f"  Date Range: {(df['start_dt'].max() - df['start_dt'].min()).days} days")

print("\nTime of Day Distribution:")
print(df['Time of Day'].value_counts().to_string())

print("\nDay of Week Distribution:")
print(df['Day of Week'].value_counts().to_string())

# Temporal patterns
print("\n\nTemporal Patterns:")
print(f"Unique dates: {df['start_dt'].dt.date.nunique()}")
print(f"Unique hours of day: {df['start_dt'].dt.hour.nunique()}")


DateTime Field Analysis
----------------------------------------

Charging Start Time:
  Earliest: 2024-01-01 00:00:00
  Latest: 2024-02-24 23:00:00
  Date Range: 54 days

Time of Day Distribution:
Time of Day
Evening      362
Morning      336
Night        312
Afternoon    310

Day of Week Distribution:
Day of Week
Saturday     205
Tuesday      200
Wednesday    197
Sunday       191
Friday       188
Monday       185
Thursday     154


Temporal Patterns:
Unique dates: 55
Unique hours of day: 24


## 6. Star Schema Mapping
Map CSV columns to dimensional model components (dimension tables and fact table) for data warehouse implementation.

In [40]:
# Map CSV columns to dimensional model (DIM and FACT tables)
print("\nStar Schema Mapping")
print("-" * 40)

star_schema = {
    "DIM_USER": ["User ID", "User Type"],
    "DIM_VEHICLE": ["Vehicle Model", "Battery Capacity (kWh)", "Vehicle Age (years)"],
    "DIM_STATION": ["Charging Station ID", "Charging Station Location", "Charger Type"],
    "DIM_TIME": ["Charging Start Time", "Time of Day", "Day of Week"],
    "DIM_WEATHER": ["Temperature (°C)"],
    "FACT_CHARGING_SESSIONS": [
        "User ID",
        "Vehicle Model",
        "Charging Station ID",
        "Charging Start Time",
        "Charging End Time",
        "Energy Consumed (kWh)",
        "Charging Duration (hours)",
        "Charging Rate (kW)",
        "Charging Cost (USD)",
        "State of Charge (Start %)",
        "State of Charge (End %)",
        "Distance Driven (since last charge) (km)"
    ]
}

for table_name, columns in star_schema.items():
    print(f"\n{table_name}:")
    for col in columns:
        print(f"  - {col}")


Star Schema Mapping
----------------------------------------

DIM_USER:
  - User ID
  - User Type

DIM_VEHICLE:
  - Vehicle Model
  - Battery Capacity (kWh)
  - Vehicle Age (years)

DIM_STATION:
  - Charging Station ID
  - Charging Station Location
  - Charger Type

DIM_TIME:
  - Charging Start Time
  - Time of Day
  - Day of Week

DIM_WEATHER:
  - Temperature (°C)

FACT_CHARGING_SESSIONS:
  - User ID
  - Vehicle Model
  - Charging Station ID
  - Charging Start Time
  - Charging End Time
  - Energy Consumed (kWh)
  - Charging Duration (hours)
  - Charging Rate (kW)
  - Charging Cost (USD)
  - State of Charge (Start %)
  - State of Charge (End %)
  - Distance Driven (since last charge) (km)


## 7. Data Cleaning Requirements
Identify data quality issues, inconsistencies, and outliers that need to be addressed before loading into the data warehouse.

In [41]:
# List data cleaning tasks based on analysis findings
print("\nData Cleaning Tasks")
print("-" * 40)

cleaning_tasks = []

# Check for missing values
missing_count = df.isnull().sum().sum()
if missing_count > 0:
    cleaning_tasks.append(f"Handle {missing_count} missing values")

# Check datetime parsing errors
if df['start_dt'].isnull().sum() > 0:
    cleaning_tasks.append("Fix datetime parsing errors in timestamp columns")

# Check for outliers in numeric fields
if (df['Charging Duration (hours)'] < 0).any():
    cleaning_tasks.append("Fix negative charging durations")

if (df['Charging Cost (USD)'] < 0).any():
    cleaning_tasks.append("Fix negative charging costs")

# Check State of Charge validity
if ((df['State of Charge (Start %)'] < 0) | (df['State of Charge (Start %)'] > 100)).any():
    cleaning_tasks.append("Fix invalid State of Charge values")

# Check data consistency
if (df['State of Charge (End %)'] < df['State of Charge (Start %)']).any():
    cleaning_tasks.append("Investigate sessions where end SoC < start SoC")

# Display tasks
if cleaning_tasks:
    print("Tasks identified:")
    for i, task in enumerate(cleaning_tasks, 1):
        print(f"  {i}. {task}")
else:
    print("No major data cleaning issues detected!")


Data Cleaning Tasks
----------------------------------------
Tasks identified:
  1. Handle 198 missing values
  2. Fix invalid State of Charge values
  3. Investigate sessions where end SoC < start SoC


## 8. Schema Recommendations
Generate Snowflake-specific recommendations for table cardinalities, indexes, data types, and partitioning strategies.

In [42]:
# Provide SQL schema recommendations based on analysis
print("\nSnowflake Schema Recommendations")
print("-" * 40)

print("\nDimension Table Cardinalities:")
print(f"  DIM_USER: ~{df['User ID'].nunique()} rows")
print(f"  DIM_VEHICLE: ~{df['Vehicle Model'].nunique()} rows")
print(f"  DIM_STATION: ~{df['Charging Station ID'].nunique()} rows")
print(f"  DIM_TIME: Generate based on date range (daily or hourly granularity)")

print("\nFact Table:")
print(f"  FACT_CHARGING_SESSIONS: {len(df)} rows initially")
print(f"  Growth rate: Depends on new charging sessions recorded")

print("\nRecommended Indexes:")
print("  - Primary keys on all dimension surrogate keys")
print("  - Foreign keys in fact table")
print("  - Clustered by Charging Start Time for time-series queries")
print("  - Bitmap indexes on categorical fields (User Type, Charger Type)")

print("\nData Types Recommendations:")
print("  - User ID: INTEGER")
print("  - Station ID: STRING/VARCHAR")
print("  - Timestamps: TIMESTAMP_LTZ (timezone aware)")
print("  - Energy/Cost metrics: FLOAT/DECIMAL")
print("  - State of Charge: INTEGER (0-100)")

print("\nPartitioning Strategy:")
print("  - Partition fact table by month (Charging Start Time)")
print("  - Consider clustering by station_id for location-based queries")


Snowflake Schema Recommendations
----------------------------------------

Dimension Table Cardinalities:
  DIM_USER: ~1320 rows
  DIM_VEHICLE: ~5 rows
  DIM_STATION: ~462 rows
  DIM_TIME: Generate based on date range (daily or hourly granularity)

Fact Table:
  FACT_CHARGING_SESSIONS: 1320 rows initially
  Growth rate: Depends on new charging sessions recorded

Recommended Indexes:
  - Primary keys on all dimension surrogate keys
  - Foreign keys in fact table
  - Clustered by Charging Start Time for time-series queries
  - Bitmap indexes on categorical fields (User Type, Charger Type)

Data Types Recommendations:
  - User ID: INTEGER
  - Station ID: STRING/VARCHAR
  - Timestamps: TIMESTAMP_LTZ (timezone aware)
  - Energy/Cost metrics: FLOAT/DECIMAL
  - State of Charge: INTEGER (0-100)

Partitioning Strategy:
  - Partition fact table by month (Charging Start Time)
  - Consider clustering by station_id for location-based queries


## 9. Save Analysis Results
Export data quality reports, numeric summaries, and sample data to the reports directory for documentation and reference.

In [43]:
# Save analysis reports into reports/ directory and print summary
print("\nSaving Analysis Results")
print("-" * 40)

# Create reports directory if required
os.makedirs("reports", exist_ok=True)

# Metadata + cleaning report
with open("reports/data_quality_report.txt", "w") as f:
    f.write("EV Charging Dataset - Data Quality Report\n")
    f.write("="*50 + "\n")
    f.write(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("Columns and Data Types:\n")
    for col, dtype in df.dtypes.items():
        f.write(f"- {col}: {dtype}\n")
    
    f.write("\nMissing Values:\n")
    missing_data = df.isnull().sum()
    for col, missing in missing_data.items():
        if missing > 0:
            f.write(f"- {col}: {missing} missing values\n")
    
    f.write("\nData Cleaning Tasks:\n")
    for task in cleaning_tasks:
        f.write(f"- {task}\n")

# Detailed numeric and sample reports
df.describe().to_csv("reports/numeric_summary.csv")
df.head(20).to_csv("reports/sample_data.csv", index=False)

# Notebook-friendly printout
print("Saved analysis reports:")
print("  reports/data_quality_report.txt")
print("  reports/numeric_summary.csv")
print("  reports/sample_data.csv")

print("\nPreview of numeric summary:")
display(df.describe().head())


Saving Analysis Results
----------------------------------------
Saved analysis reports:
  reports/data_quality_report.txt
  reports/numeric_summary.csv
  reports/sample_data.csv

Preview of numeric summary:


,Battery Capacity (kWh),Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Vehicle Age (years),start_dt,end_dt
count,1320.000000,1254.000000,1320.000000,1254.000000,1320.000000,1320.000000,1320.000000,1254.000000,1320.000000,1320.000000,1320,1320
mean,74.534692,42.642894,2.269377,25.963003,22.551352,49.130012,75.141590,153.596788,15.263591,3.612843,2024-01-28 11:30:00,2024-01-28 13:43:30.863636480
min,1.532807,0.045772,0.095314,1.472549,0.234317,2.325959,7.604224,0.862361,-10.724770,0.000000,2024-01-01 00:00:00,2024-01-01 00:39:00
25%,62.000000,23.881193,1.397623,13.856583,13.368141,27.786903,62.053266,79.445335,2.800664,2.000000,2024-01-14 17:45:00,2024-01-14 19:34:30
50%,75.000000,42.691405,2.258136,25.603799,22.076360,48.241771,75.682496,152.259867,14.630846,4.000000,2024-01-28 11:30:00,2024-01-28 14:07:00
